In [6]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, RidgeCV

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


df = pd.read_csv("../data/forest+fires/forestfires.csv", delimiter=",")
print("Shape: ", df.shape)
df.info()
print(df.describe(include="all").T)
print(df.isna().mean().sort_values(ascending=False))
print(df.nunique().sort_values())
print(df[df.duplicated()])
print("Duplicates: ", df.duplicated().sum())


Shape:  (517, 13)
<class 'pandas.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X       517 non-null    int64  
 1   Y       517 non-null    int64  
 2   month   517 non-null    str    
 3   day     517 non-null    str    
 4   FFMC    517 non-null    float64
 5   DMC     517 non-null    float64
 6   DC      517 non-null    float64
 7   ISI     517 non-null    float64
 8   temp    517 non-null    float64
 9   RH      517 non-null    int64  
 10  wind    517 non-null    float64
 11  rain    517 non-null    float64
 12  area    517 non-null    float64
dtypes: float64(8), int64(3), str(2)
memory usage: 52.6 KB
       count unique  top freq        mean         std   min    25%    50%  \
X      517.0    NaN  NaN  NaN    4.669246    2.313778   1.0    3.0    4.0   
Y      517.0    NaN  NaN  NaN    4.299807      1.2299   2.0    4.0    4.0   
month    517     12  aug  184         NaN  

In [2]:
categorical_columns = ["month", "day"]
numerical_columns = ['X', 'Y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']

X = df[categorical_columns + numerical_columns]
y = df["area"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)






In [3]:
#Setting up Preprocessing Pipeline
categorical_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1
)
numerical_pipe = SimpleImputer(strategy="mean")

preprocessing = ColumnTransformer(
    [
        ("cat", categorical_encoder, categorical_columns),
        ("num", numerical_pipe, numerical_columns),
    ],
    verbose_feature_names_out=False,
)

In [4]:
# Trying Linear Regression Model
model = make_pipeline(preprocessing, LinearRegression())
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = root_mean_squared_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("Mean Absolute Error: ", mae, "\nRoot Mean Squared Error: ", rmse,"\nR2 Score: ", r2)

Mean Absolute Error:  21.7513220462986 
Root Mean Squared Error:  97.06194040967341 
R2 Score:  0.0056120475921142


In [5]:
# Trying Ridge Model
model = make_pipeline(preprocessing, RidgeCV(alphas=[0.01,0.1,1,10,100]))
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = root_mean_squared_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("Mean Absolute Error: ", mae, "\nRoot Mean Squared Error: ", rmse,"\nR2 Score: ", r2)

Mean Absolute Error:  21.7223270791566 
Root Mean Squared Error:  97.07130923625378 
R2 Score:  0.005420073315438478
